# AUX / QC inspection — earthquake-cycle clusters

Ports the per-cluster QC notebooks into the unified framework, importing the *same*
functions the pipeline uses:

1. **Station misorientation** (`core.misorientation`) — time-dependent PCA angle per used station.
2. **ZRT rotation** (`core.rotation`) — N/E → radial/transverse using `(baz − misorientation)`.
3. **Waveform similarity** (`analysis.similarity`) — NCC of each event vs a template at a
   reference station; high/stable NCC ⇒ a repeating (tightly-clustered) source.

These are diagnostics: they are **not** relocation inputs and not part of regression (no frozen
baseline artifacts). Set `CLUSTER` and Run-All. Requires a completed run tree
(`runs/<cluster>/waveforms_100km` with picks) — e.g. after `run_pipeline --through dtcc`.

In [ ]:
import os
os.environ.setdefault("PYTHONPATH", "/home/msseo/works/10.Earthquake_cycle_project")
import sys; sys.path.insert(0, "/home/msseo/works/10.Earthquake_cycle_project")
from glob import glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from obspy import read

from pipeline import config
from pipeline.core import misorientation, rotation, waveforms
from pipeline.analysis import similarity

CLUSTER = "gwangyang"        # gwangyang | kimcheon | jangsung | gyeongju
cfg = config.load_cluster(CLUSTER)
catalog = waveforms.load_catalog(cfg)
print(f"{cfg.name}: {len(catalog)} events, region {cfg.region}, wf_source={cfg.wf_source}")

## 1. Station misorientation

Per-sensor PCA tables give each station's orientation error over time. We attach the angle valid at each event's date and write `used_stations_100km_event<i>.csv`.

In [ ]:
counts = misorientation.run_misorientation(cfg)
print("stations with a misorientation angle, per event:", counts)
ev1 = catalog[0]["event_id"]
t = pd.read_csv(os.path.join(config.station_table_dir(cfg), "used_stations_100km_event1.csv"))
display(t[["Network","Code","Sensor","Misorientation"]].head(10))

m = t.dropna(subset=["Misorientation"]).sort_values("Misorientation")
fig, ax = plt.subplots(figsize=(13, 4))
colors = {"HH":"tab:blue","HG":"tab:orange","EL":"tab:green"}
ax.bar(m["Code"], m["Misorientation"], color=[colors.get(s,"gray") for s in m["Sensor"]])
ax.axhline(0, color="k", lw=0.6); ax.set_ylabel("misorientation (deg)")
ax.set_title(f"{cfg.name} — station misorientation (event 1, {ev1})")
ax.tick_params(axis="x", rotation=90, labelsize=7)
handles = [plt.Rectangle((0,0),1,1,color=colors[s]) for s in colors]
ax.legend(handles, colors.keys(), title="sensor"); plt.tight_layout(); plt.show()

## 2. ZRT rotation

Rotate N/E → R/T with `ba = (baz − misorientation) % 360` and show a 3-component record section (Z / R / T) for the nearest few stations of one event.

In [ ]:
ev = catalog[0]
used = pd.read_csv(config.used_stations_csv(cfg)); sof = dict(zip(used.Code, used.Sensor))
tables = misorientation._load_tables(cfg)
n = rotation.rotate_event(cfg, ev, sof, tables)
print(f"rotated {n} stations for event {ev['event_id']} -> runs/{cfg.name}/rotated/")

rdir = os.path.join(config.run_root(cfg), "rotated", ev["event_id"])
# nearest stations (by SAC dist on Z)
zs = []
for z in glob(os.path.join(rdir, "*Z.sac")):
    s = read(z)[0].stats.sac
    zs.append((s.get("dist", 1e9), os.path.basename(z)))
zs.sort()
pick = [b for _, b in zs[:6]]
fig, axes = plt.subplots(len(pick), 3, figsize=(14, 1.6*len(pick)), sharex=True)
for r, zname in enumerate(pick):
    code = zname.split(".")[2]; sensor = zname.split(".")[3][:2]
    for col, comp in enumerate(["Z","R","T"]):
        f = os.path.join(rdir, zname.replace(f"{sensor}Z.sac", f"{sensor}{comp}.sac"))
        ax = axes[r, col]
        if os.path.exists(f):
            tr = read(f)[0].detrend("demean").normalize()
            ax.plot(tr.times(), tr.data, lw=0.5, color="k")
        if r == 0: ax.set_title(comp)
        if col == 0: ax.set_ylabel(code, fontsize=8, rotation=0, ha="right", va="center")
        ax.set_yticks([])
axes[-1,1].set_xlabel("time from window start (s)")
fig.suptitle(f"{cfg.name} — ZRT record section, event {ev['event_id']} (nearest 6)")
plt.tight_layout(); plt.show()

## 3. Waveform similarity

NCC of every event's phase window vs a template event (default the largest magnitude) at one reference station. Tight, repeating sources stay highly similar across the sequence.

In [ ]:
df = similarity.similarity(cfg, comp="Z")
print(f"reference station: {df.attrs.get('station')}  |  template event: {df.attrs.get('template')}  |  {len(df)} events")
display(df[["event_id","time","mag","ncc","shift_s","p_rms"]])

if len(df) >= 2:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].scatter(df["mag"], df["ncc"], c="tab:red"); ax[0].set_xlabel("magnitude"); ax[0].set_ylabel("NCC vs template")
    ax[0].set_title("similarity vs magnitude"); ax[0].set_ylim(0, 1.05); ax[0].grid(alpha=.3)
    ax[1].plot(df["time"], df["ncc"], "o-", color="tab:blue"); ax[1].set_xlabel("origin time (UTC)")
    ax[1].set_ylabel("NCC vs template"); ax[1].set_title("similarity over time"); ax[1].set_ylim(0,1.05); ax[1].grid(alpha=.3)
    fig.autofmt_xdate(); plt.tight_layout(); plt.show()

In [ ]:
# Overlay the template vs the most- and least-similar events at the reference station
if len(df) >= 2:
    sta = df.attrs["station"]; sensor = dict(zip(used.Code, used.Sensor)).get(sta)
    hi = df.loc[df["ncc"].idxmax(), "event_id"]; lo = df.loc[df["ncc"].idxmin(), "event_id"]
    fig, ax = plt.subplots(figsize=(13, 4))
    for eid, col, lab in [(df.attrs["template"],"k","template"), (hi,"tab:green",f"max NCC {hi}"),
                          (lo,"tab:red",f"min NCC {lo}")]:
        fs = glob(os.path.join(config.event_wf_dir(cfg, eid), f"{eid}.*.{sta}.{sensor}Z.sac"))
        if not fs: continue
        tr = read(fs[0])[0]; pk = tr.stats.sac.get("a", -12345.0)
        if pk == -12345.0: continue
        pt = tr.stats.starttime - tr.stats.sac.b + pk
        w = tr.copy().detrend("demean").taper(0.05).filter("highpass", freq=1, corners=6, zerophase=True)\
              .slice(pt-0.5, pt+3.5).detrend("demean").normalize()
        ax.plot(w.times(), w.data, color=col, lw=1, label=lab, alpha=.8)
    ax.legend(); ax.set_xlabel("time from P−0.5 s"); ax.set_title(f"{cfg.name} — P waveforms at {sta} (Z)")
    plt.tight_layout(); plt.show()

### Interpretation

- **Misorientation** corrects sensor azimuth so R/T are physically meaningful; large angles flag
  stations needing care in any polarity/amplitude analysis.
- **ZRT rotation** concentrates S energy on T (and P/SV on R), a basic QC of orientation + picks.
- **Waveform similarity** near 1.0 across the sequence confirms a compact, repeating source — the
  premise that makes HypoDD cross-correlation (`dt.cc`) relocation meaningful. A sharp NCC drop or a
  systematic amplitude/time trend would flag a waveform change (e.g. a different sub-fault or a
  mislabelled event) worth inspecting before trusting the relative locations.